<div dir=rtl>

# مینی‌پروژه‌ی تلفیقی مهندسی — دفترچه‌ی پرسش‌ها

## پیش‌بینی بار، خوشه‌بندی فیدرها، نقشه‌ی SOM و تحلیل تابلوهای سرعت

این دفترچه نسخه‌ی دانشجویی است. کد تولید داده در این دفترچه قرار نگرفته است؛ داده‌ها را با فایل مستقل تولیدکننده بسازید.

### فایل‌های ورودی
- electrical_load_timeseries.csv: داده‌ی ساعتی بار الکتریکی و دما، همراه با داده‌های گمشده، پرت، تکراری و زمان‌های حذف‌شده.
- electrical_profiles.csv: ویژگی‌های خلاصه‌ی ۲۴۰ فیدر با چهار نوع پروفایل مرجع؛ دارای داده‌ی گمشده، پرت و ردیف تکراری.
- traffic_sign_dataset.csv: تصویرهای مصنوعی تابلو سرعت ۵۰. مجموعه‌ی آموزش فقط تصویرهای عادی دارد؛ مجموعه‌ی آزمون شامل تصویرهای عادی و دست‌کاری‌شده است. هر تصویر به‌صورت ۳۲×۳۲ شدت خاکستری ذخیره شده است.

### نمره‌دهی پایه
| بخش | امتیاز |
|---|---:|
| کدنویسی و خروجی سری زمانی | ۱۵ |
| کدنویسی K-Means | ۸ |
| کدنویسی SOM | ۷ |
| کدنویسی Hopfield | ۱۵ |
| **جمع کدنویسی** | **۴۵** |
| تئوری سری زمانی | ۵ |
| تئوری K-Means | ۵ |
| تئوری SOM | ۵ |
| تئوری Hopfield | ۵ |
| **جمع تئوری** | **۲۰** |
| **کدنویسی + تئوری** | **۶۵** |
| ارائه‌ی نهایی | ۳۵ |
| **نمره‌ی پایه‌ی کل** | **۱۰۰** |
| امتیاز تشویقی | **۱۰ امتیاز اضافه** |

امتیاز تشویقی جدا از ۱۰۰ امتیاز پایه محاسبه می‌شود. بنابراین بیشترین امتیاز ممکن با تشویقی ۱۱۰ است.

### شناسه‌ی پاسخ‌ها
پاسخ‌های عملی با شماره‌هایی مانند 1.1 و 2.2 مشخص می‌شوند. پاسخ‌های تئوری با پسوند T، مانند 1.1T، مشخص می‌شوند. پاسخ هر قسمت باید در همان بخش دفترچه قرار بگیرد.

### آماده‌سازی
از پوشه‌ی اصلی پروژه اجرا کنید:
    
    python generate_project_data.py --seed 12345678 --output notebooks/data

سپس دفترچه را از پوشه‌ی notebooks باز کنید. در صورت استفاده از seed دیگر، همان seed را برای تولید همه‌ی فایل‌ها به‌کار ببرید.

وابستگی‌ها:
    
    pip install numpy pandas matplotlib scikit-learn minisom pillow

**محدودیت علمی پروژه‌ی Hopfield:** مسئله‌ی این بخش خواندن عمومی هر نوع تابلو یا OCR نیست. شبکه یک الگوی مرجع تمیز از تابلوی ۵۰ را به‌عنوان حافظه نگه می‌دارد و میزان تفاوت تصویر ورودی و الگوی بازیابی‌شده را برای غربالگری تغییرات مصنوعی اندازه می‌گیرد. این آزمایش به‌تنهایی اصالت یا جعلی‌بودن یک تابلوی واقعی را اثبات نمی‌کند.

</div>

<div dir=rtl>

## بخش ۱ — پیش‌بینی بار الکتریکی با سری زمانی (۱۵ امتیاز کدنویسی)

هدف، پیش‌بینی بار ساعت آینده است. داده‌ی خام را پیش از هر پاک‌سازی بررسی کنید. پاک‌سازی باید با منطق فیزیکی و زمانی توجیه شود؛ فقط حذف سطرهای ناقص کافی نیست.

### 1.1 — بررسی داده‌ی خام (۲ امتیاز)
فایل را بخوانید و ابعاد، نوع داده‌ها، تعداد مقادیر گمشده، timestampهای تکراری و مقادیر غیرواقعی بار و دما را گزارش کنید. چند ردیف نمونه را نمایش دهید.

</div>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, silhouette_score,
    adjusted_rand_score, confusion_matrix, precision_score,
    recall_score, f1_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from minisom import MiniSom

DATA_DIR = Path("data")
load_raw = pd.read_csv(DATA_DIR / "electrical_load_timeseries.csv")
profiles_raw = pd.read_csv(DATA_DIR / "electrical_profiles.csv")
signs_raw = pd.read_csv(DATA_DIR / "traffic_sign_dataset.csv")

print("Load data:", load_raw.shape)
print("Electrical profiles:", profiles_raw.shape)
print("Traffic sign data:", signs_raw.shape)
display(load_raw.head())
display(profiles_raw.head())
display(signs_raw.iloc[:5, :8])

Load data: (2165, 3)
Electrical profiles: (244, 9)
Traffic sign data: (1000, 1028)


,timestamp,load_kw,temperature_c
0,2025-03-06 03:00:00,208.085620,24.794243
1,2025-01-02 22:00:00,203.384609,22.162815
2,2025-03-20 23:00:00,225.540003,15.145669
3,2025-03-16 00:00:00,187.094371,19.769358
4,2025-02-03 21:00:00,249.732911,27.813952


,feeder_id,mean_load_kw,std_load_kw,peak_load_kw,load_factor,night_energy_ratio,peak_to_mean,variability_index,reference_profile
0,F037,175.500347,18.338993,217.348298,0.855990,0.374317,1.238449,0.104495,Base-load
1,F155,146.750933,33.279299,249.332876,0.608351,0.301276,1.699021,0.226774,Variable
2,F199,156.326879,31.572359,234.606104,0.722678,0.301391,1.500741,0.201964,Variable
3,F045,199.812614,11.319777,236.175483,0.866036,0.414387,1.181985,0.056652,Base-load
4,F195,147.138624,37.791146,228.933312,0.641909,0.279198,1.555902,0.256840,Variable


,image_id,split,label,alteration,pixel_00_00,pixel_00_01,pixel_00_02,pixel_00_03
0,S0001,train,0,normal,247,247,247,246
1,S0002,train,0,normal,252,253,253,253
2,S0003,train,0,normal,247,247,247,247
3,S0004,train,0,normal,254,254,254,254
4,S0005,train,0,normal,250,250,250,249


<div dir=rtl>

### 1.2 — پاک‌سازی زمان‌محور (۳ امتیاز)
timestamp را به datetime تبدیل کنید، رکوردهای تکراری هر ساعت را با روشی مناسب تجمیع کنید، شاخص ساعتی پیوسته بسازید و مقادیر نامعتبر را به مقدار گمشده تبدیل کنید. برای پرکردن مقادیر گمشده از روشی استفاده کنید که برای پیش‌بینی آینده قابل دفاع باشد. تعداد رکوردهای تکراری، زمان‌های اضافه‌شده و مقادیر اصلاح‌شده را گزارش کنید.

</div>

In [2]:
# TODO 1.2: parse timestamps, validate physical ranges, aggregate duplicate hours,
# reindex to an hourly timeline, and impute missing values using past information.


<div dir=rtl>

### 1.3 — مقایسه‌ی ورودی خام و پاک‌شده (۲ امتیاز)
نمودار بار بر حسب زمان را پیش از پاک‌سازی و پس از پاک‌سازی رسم کنید. نقاط پرت، شکاف‌ها و اثر روش پاک‌سازی خود را توضیح دهید.

</div>

In [3]:
# TODO 1.3: plot raw and cleaned load against time.


<div dir=rtl>

### 1.4 — ویژگی‌سازی و تقسیم زمانی (۳ امتیاز)
ویژگی‌های lag-1، lag-24 و lag-168 و یک میانگین متحرک ۲۴ ساعته بسازید. میانگین متحرک باید فقط از گذشته استفاده کند. داده را به ترتیب زمان به آموزش و آزمون تقسیم کنید؛ تقسیم تصادفی مجاز نیست. پیش‌بینی ساده‌ی lag-24 را به‌عنوان خط مبنا محاسبه کنید.

</div>

In [4]:
# TODO 1.4: create lag/rolling/calendar features and a chronological train/test split.


<div dir=rtl>

### 1.5 — مدل و ارزیابی (۵ امتیاز)
یک مدل رگرسیون مناسب آموزش دهید و آن را با خط مبنای lag-24 مقایسه کنید. MAE و RMSE هر دو را گزارش دهید و نمودار مقدار واقعی و پیش‌بینی‌شده در بازه‌ی آزمون را رسم کنید. نتیجه را با توجه به نوسان بار تفسیر کنید.

</div>

In [5]:
# TODO 1.5: train a forecasting model, compute MAE/RMSE for model and baseline,
# and plot the test interval predictions.


<div dir=rtl>

### 1.1T — نشت اطلاعات و تقسیم زمانی (۲٫۵ امتیاز تئوری)
توضیح دهید چرا تقسیم تصادفی برای این مسئله مناسب نیست. یک مثال مشخص از نشت اطلاعات هنگام ساخت lag یا rolling feature ارائه کنید و بگویید چگونه از آن جلوگیری می‌کنید.

### 1.2T — خط مبنا و معیار خطا (۲٫۵ امتیاز تئوری)
چرا مقایسه با lag-24 ضروری است؟ تفاوت MAE و RMSE چیست و در چه شرایطی RMSE برای خطاهای پیک اهمیت بیشتری پیدا می‌کند؟

</div>

<div dir=rtl>

## بخش ۲ — K-Means برای خوشه‌بندی پروفایل فیدرها (۸ امتیاز کدنویسی)

هر ردیف یک پروفایل فیدر است. ستون feeder_id شناسه و reference_profile برچسب مرجع مصنوعی است. این دو ستون را به مدل ندهید. reference_profile فقط پس از خوشه‌بندی برای تفسیر و ارزیابی پسینی مجاز است.

### 2.1 — پاک‌سازی و آماده‌سازی (۲ امتیاز)
مقادیر گمشده، تکراری و پرت را شناسایی کنید. محدوده‌های فیزیکی ویژگی‌ها را بررسی کنید، داده را پاک‌سازی کنید و همه‌ی ویژگی‌های مدل را استاندارد کنید. یک نمودار دوبعدی خام و یک نمودار دوبعدی پاک‌شده رسم کنید.

</div>

In [6]:
# TODO 2.1: exclude feeder_id and reference_profile from X, clean invalid values,
# handle duplicates/missingness, scale the features, and visualize raw/clean inputs.


<div dir=rtl>

### 2.2 — انتخاب K (۳ امتیاز)
حداقل سه مقدار K را مقایسه کنید. نمودار inertia و امتیاز silhouette را رسم کنید و K نهایی را با استدلال انتخاب کنید. فقط به کم‌شدن inertia اکتفا نکنید.

</div>

In [7]:
# TODO 2.2: evaluate several K values with inertia and silhouette.


<div dir=rtl>

### 2.3 — مدل نهایی و تفسیر خوشه‌ها (۳ امتیاز)
K-Means نهایی را آموزش دهید، خوشه‌ها و centroidها را در یک نمای دوبعدی نمایش دهید و ویژگی‌های هر خوشه را تفسیر کنید. در صورت استفاده از reference_profile برای محاسبه‌ی ARI یا جدول توافق، صریحاً بنویسید که این برچسب در آموزش مدل استفاده نشده است.

</div>

In [8]:
# TODO 2.3: fit final K-Means, visualize clusters/centroids, and interpret them.


<div dir=rtl>

### 2.1T — مقیاس‌بندی و انتخاب K (۳ امتیاز تئوری)
توضیح دهید چرا استانداردسازی برای K-Means اهمیت دارد. inertia و silhouette چه چیزی را اندازه می‌گیرند؟ چرا انتخاب K فقط با کمینه‌کردن inertia اشتباه است؟

### 2.2T — فرض‌ها و محدودیت‌ها (۲ امتیاز تئوری)
دو محدودیت K-Means را در خوشه‌بندی فیدرها توضیح دهید. چرا شماره‌ی خوشه‌ها معنای ذاتی ندارد؟ نقش centroid در تفسیر خوشه چیست؟

</div>

<div dir=rtl>

## بخش ۳ — SOM برای نگاشت توپولوژیک همان پروفایل‌ها (۷ امتیاز کدنویسی)

از همان ویژگی‌های پاک‌سازی‌شده و استانداردشده‌ی بخش K-Means استفاده کنید. SOM یک شبکه‌ی دوبعدی از بردارهای وزن است؛ نورون برنده و همسایه‌های آن در طول آموزش به ورودی نزدیک می‌شوند.

### 3.1 — آموزش SOM (۲ امتیاز)
یک شبکه‌ی دوبعدی با ابعاد انتخابی بسازید. نرخ یادگیری، اندازه‌ی همسایگی و تعداد گام‌های آموزش را مشخص کنید. انتخاب‌ها را کوتاه توجیه کنید.

</div>

In [9]:
# TODO 3.1: train a MiniSom model using the standardized profile matrix.


<div dir=rtl>

### 3.2 — BMU، اشغال نقشه و U-Matrix (۳ امتیاز)
برای هر فیدر نورون برنده (BMU) را بیابید. نقشه‌ی تعداد نمونه‌های هر نورون و U-Matrix را رسم کنید. نواحی متراکم و مرزهای احتمالی بین گروه‌ها را تفسیر کنید.

</div>

In [10]:
# TODO 3.2: compute BMUs, occupancy map, and U-Matrix; visualize both maps.


<div dir=rtl>

### 3.3 — مقایسه با K-Means (۲ امتیاز)
جایگاه BMUها را با رنگ خوشه‌های K-Means نمایش دهید و خطای کوانتیزاسیون SOM را گزارش کنید. توضیح دهید چرا خروجی SOM الزاماً همان برچسب‌های K-Means نیست.

</div>

In [11]:
# TODO 3.3: compare SOM map locations with K-Means clusters and report quantization error.


<div dir=rtl>

### 3.1T — الگوریتم و ابرپارامترها (۳ امتیاز تئوری)
مراحل انتخاب BMU و به‌روزرسانی همسایه‌ها را توضیح دهید. اثر نرخ یادگیری و شعاع همسایگی را در ابتدای آموزش و انتهای آن شرح دهید. اگر شعاع خیلی سریع کوچک شود چه خطری وجود دارد؟

### 3.2T — تفسیر نقشه (۲ امتیاز تئوری)
U-Matrix و نقشه‌ی اشغال چه اطلاعات متفاوتی می‌دهند؟ دو تفاوت مهم SOM و K-Means را از نظر هدف و خروجی بیان کنید.

</div>

<div dir=rtl>

## بخش ۴ — Hopfield برای غربالگری تصویر تابلو سرعت (۱۵ امتیاز کدنویسی)

در داده‌ی این بخش، تصویر عادی یک تابلوی سرعت ۵۰ است. بخشی از تصویرهای آزمون به شکل مصنوعی دست‌کاری شده‌اند: عدد تغییر کرده، خط یا مانع اضافه شده یا بخشی از عدد پوشانده شده است. برچسب ۱ به معنی «دست‌کاری‌شده در داده‌ی مصنوعی» و صفر به معنی «عادی» است. این برچسب‌ها را برای آموزش شبکه استفاده نکنید؛ شبکه باید فقط از نمونه‌های عادی آموزش ببیند.

### 4.1 — بررسی داده و نمایش تصویرها (۲ امتیاز)
ابعاد داده، تعداد نمونه‌های هر split و هر کلاس را گزارش کنید. چند نمونه‌ی عادی و دست‌کاری‌شده را به‌صورت تصویر ۳۲×۳۲ نمایش دهید.

</div>

In [12]:
# TODO 4.1: identify pixel columns, inspect split/label counts, and display examples.


<div dir=rtl>

### 4.2 — تبدیل پیکسل و انتخاب حافظه (۲ امتیاز)
شدت پیکسل‌ها را به بازه‌ی صفر تا یک ببرید و آن‌ها را با آستانه‌ی باینری‌سازی به حالت‌های bipolar یعنی -1 و +1 تبدیل کنید. از بخشی از نمونه‌های عادی آموزش، یک prototype نماینده انتخاب کنید؛ باقی نمونه‌های عادی آموزش را برای کالیبراسیون نگه دارید. برچسب‌های آزمون نباید در انتخاب prototype یا کالیبراسیون دخالت کنند.

</div>

In [13]:
# TODO 4.2: normalize pixels, binarize with a configurable cutoff, choose a clean medoid,
# and reserve the remaining normal training images for calibration.


<div dir=rtl>

### 4.3 — ساخت و بازیابی شبکه (۳ امتیاز)
وزن‌ها را با قاعده‌ی Hebbian برای prototype انتخاب‌شده بسازید. ماتریس وزن باید متقارن و قطر آن صفر باشد. بازیابی را به‌صورت ناهم‌زمان پیاده‌سازی کنید و برای یک نمونه، تغییر انرژی را نمایش دهید.

</div>

In [14]:
# TODO 4.3: implement fit, energy, and asynchronous recall for a binary Hopfield network.


<div dir=rtl>

### 4.4 — کالیبراسیون آستانه‌ی هشدار (۲ امتیاز)
برای نمونه‌های عادی کالیبراسیون، اختلاف درصدی بین ورودی و خروجی بازیابی‌شده را به‌عنوان anomaly score محاسبه کنید. آستانه‌ی هشدار را فقط از همین نمونه‌های عادی تعیین کنید؛ برای مثال صدک ۹۵.

</div>

In [15]:
# TODO 4.4: calculate calibration anomaly scores and select an alert threshold.


<div dir=rtl>

### 4.5 — ارزیابی و مقایسه‌ی cutoffها (۴ امتیاز)
دو آستانه‌ی باینری‌سازی پیکسل 0.4 و 0.5 را آزمایش کنید. برای هرکدام threshold هشدار را مجدداً فقط با داده‌ی عادی کالیبراسیون تعیین کنید و روی مجموعه‌ی آزمون، precision، recall، F1 و specificity را گزارش دهید. ماتریس درهم‌ریختگی یا جدول TP/FP/TN/FN را نیز ارائه کنید.

دقت کنید: cutoff پیکسل، شدت خاکستری را به -1/+1 تبدیل می‌کند؛ threshold هشدار، anomaly score را به برچسب عادی/مشکوک تبدیل می‌کند. این دو آستانه یک مفهوم نیستند.

</div>

In [16]:
# TODO 4.5: compare pixel cutoffs 0.4 and 0.5 and report test-set F1 and other metrics.


<div dir=rtl>

### 4.6 — مقایسه و نمایش نمونه‌ها (۲ امتیاز)
خروجی Hopfield را با baseline ساده‌ی فاصله‌ی Hamming تا prototype مقایسه کنید. چند نمونه‌ی درست، هشدار اشتباه و دست‌کاری‌شده‌ی تشخیص‌داده‌شده را کنار ورودی و خروجی بازیابی‌شده نمایش دهید. محدودیت روش را بیان کنید.

</div>

In [17]:
# TODO 4.6: compare with direct template matching and visualize representative test cases.


<div dir=rtl>

### 4.1T — حافظه و انرژی Hopfield (۲٫۵ امتیاز تئوری)
رابطه‌ی Hebbian برای یک الگوی bipolar را بنویسید و نقش تقارن ماتریس وزن و صفر بودن قطر را توضیح دهید. چرا به‌روزرسانی ناهم‌زمان با وزن‌های متقارن انرژی را افزایش نمی‌دهد؟ اگر چند الگوی بسیار مشابه ذخیره شوند چه مشکلی ممکن است رخ دهد؟

### 4.2T — آستانه و F1 (۲٫۵ امتیاز تئوری)
نتایج F1 برای cutoffهای پیکسل 0.4 و 0.5 را تحلیل کنید. اگر cutoff بزرگ‌تر شود، چه پیکسل‌هایی تغییر حالت می‌دهند؟ چرا threshold هشدار باید از داده‌ی عادی کالیبره شود؟ چه زمانی precision یا recall را مهم‌تر می‌دانید؟

</div>

<div dir=rtl>

## امتیاز تشویقی — ۱۰ امتیاز اضافه

این پرسش‌ها دشوارترند و نیازمند تحلیل اثر ابرپارامترها و محدودیت مدل هستند. امتیاز آن‌ها جدا از ۱۰۰ امتیاز پایه است.

### 1.B — سری زمانی (۳ امتیاز اضافه)
یک ablation طراحی کنید: تعداد lagها، طول rolling window یا عمق/تعداد درخت‌های Random Forest را تغییر دهید. هر بار فقط یک عامل را تغییر دهید، از اعتبارسنجی زمانی استفاده کنید و نشان دهید آیا بهبود نسبت به baseline پایدار است یا خیر.

### 2.B و 3.B — K-Means و SOM (۴ امتیاز اضافه؛ هرکدام ۲ امتیاز)
- K-Means: خوشه‌بندی را با داده‌ی استانداردنشده هم اجرا کنید و تغییر خوشه‌ها، silhouette و centroidها را با نسخه‌ی استانداردشده مقایسه کنید.
- SOM: ابعاد شبکه یا نرخ یادگیری/شعاع همسایگی را تغییر دهید. اثر آن را بر quantization error و توپولوژی نقشه توضیح دهید.

### 4.B — Hopfield (۳ امتیاز اضافه)
آزمایش پایداری انجام دهید: روی تصویرهای عادی، تغییر روشنایی و جابه‌جایی کوچک هندسی ایجاد کنید یا نمونه‌های بیشتری از الگوهای حافظه اضافه کنید. راهکاری برای کاهش هشدار کاذب پیشنهاد و با F1 و specificity ارزیابی کنید. توضیح دهید آیا راهکار شما واقعاً قابلیت تعمیم را بهتر کرده یا فقط داده‌ی مصنوعی را ساده‌تر کرده است.

</div>

<div dir=rtl>

## بخش پنجم — ارائه‌ی نهایی (۳۵ امتیاز)

| معیار | امتیاز |
|---|---:|
| توضیح مسئله و منطق روش‌ها | ۸ |
| نمایش داده‌ی خام، پاک‌سازی و توجیه تصمیم‌ها | ۸ |
| نمودارهای ورودی و خروجی و تفسیر آن‌ها | ۹ |
| تحلیل نتایج، خطاها و محدودیت‌ها | ۶ |
| انسجام، پاسخ به پرسش‌ها و مشارکت اعضا | ۴ |
| **جمع** | **۳۵** |

### تحویل
- دفترچه‌ی دانشجویی اجراشده با پاسخ‌های شماره‌گذاری‌شده.
- نمودارهای داده‌ی خام و پاک‌سازی‌شده، به‌علاوه‌ی نمودارها و معیارهای مدل.
- پاسخ تئوری در بخش‌های شماره‌گذاری‌شده با پسوند T.
- فایل‌های CSV تولیدشده با seed گروه.
- فایل ارائه‌ی نهایی.

در بخش Hopfield، نتیجه را به‌عنوان غربالگری تغییرات مصنوعی گزارش کنید؛ ادعای تشخیص جعل در تابلوهای واقعی نداشته باشید.

</div>